# 05 -- dim_nfl_players Seed

**Purpose**: Build `dim_nfl_players` — the **central player registry** for
the fantasy league. Every player on a fantasy roster must have a row here.
Veterans are populated from nflverse at season start. Rookies are added
post-signing via ETL enrichment from `dim_rookie_prospect` (joined on `pfr_id`).

**Key**: `gsis_id` (NFL-assigned; the universal player FK for all fact tables)

**Relationship to dim_rookie_prospect**:
- Pre-signing: prospects exist only in `dim_rookie_prospect` (no `gsis_id` yet)
- Post-signing: nflverse assigns `gsis_id`; ETL merges the prospect row here
  via `pfr_id` and the `dim_rookie_prospect.gsis_id` field is back-populated

**Outputs**:
- `data/dim_nfl_players.parquet`

In [1]:
import pandas as pd
import numpy as np
import nflreadpy as nfl
from pathlib import Path
from datetime import datetime, date
from dataclasses import dataclass, field


@dataclass
class LeagueConfig:
    draft_year: int = 2026
    total_cap: int = 500_000_000
    num_teams: int = 28
    num_conferences: int = 2
    initial_contract_years: int = 3
    extension_contract_years: int = 3
    fa_minimum_salary: int = 2_000_000
    data_dir: str = "data"
    fuzzy_auto_threshold: int = 90
    fuzzy_review_threshold: int = 70


CFG = LeagueConfig()
DATA = Path(CFG.data_dir)
TODAY = date.today().isoformat()
DATA.mkdir(exist_ok=True)

## 1 -- Load Raw Players

In [2]:
# -- Load players from nflreadpy ------------------------------------------------
print("Loading NFL players from nflverse...")
try:
    players_raw = nfl.load_players().to_pandas()
except Exception as e:
    raise RuntimeError(f"Failed to load nflverse players: {e}") from e

print(f"Raw rows: {len(players_raw):,}")
print(f"Columns ({len(players_raw.columns)}): {list(players_raw.columns)}")

Loading NFL players from nflverse...
Raw rows: 24,966
Columns (39): ['gsis_id', 'display_name', 'common_first_name', 'first_name', 'last_name', 'short_name', 'football_name', 'suffix', 'esb_id', 'nfl_id', 'pfr_id', 'pff_id', 'otc_id', 'espn_id', 'smart_id', 'birth_date', 'position_group', 'position', 'ngs_position_group', 'ngs_position', 'height', 'weight', 'headshot', 'college_name', 'college_conference', 'jersey_number', 'rookie_season', 'last_season', 'latest_team', 'status', 'ngs_status', 'ngs_status_short_description', 'years_of_experience', 'pff_position', 'pff_status', 'draft_year', 'draft_round', 'draft_pick', 'draft_team']


## 2 -- Select Columns and Save

In [ ]:
# -- Select + rename nflverse columns to canonical dim schema -----------------
# nflverse load_players() column names differ from this project's canonical
# schema. Map desired_output_col -> actual source column so values are pulled
# from the RIGHT column (the old name-only select created empty columns when the
# wished-for name was absent). Key corrections:
#   team_abbr   <- latest_team   (nflverse names current team "latest_team")
#   entry_year  <- rookie_season (season the player entered the NFL)
#   draft_club  <- draft_team
#   draft_number<- draft_pick
# Cross-ref IDs: this build provides esb_id/nfl_id/otc_id/smart_id (NOT the
# yahoo/sleeper/rotowire/etc. that the old list wished for).
_COLMAP = {
    # Identity
    "gsis_id":             "gsis_id",            # primary key (NFL GSIS)
    "display_name":        "display_name",
    "first_name":          "first_name",
    "last_name":           "last_name",
    "birth_date":          "birth_date",
    # Current roster
    "status":              "status",
    "team_abbr":           "latest_team",
    "position":            "position",
    "position_group":      "position_group",
    "jersey_number":       "jersey_number",
    "years_of_experience": "years_of_experience",
    # College / draft
    "college_name":        "college_name",
    "college_conference":  "college_conference",
    "entry_year":          "rookie_season",
    "last_season":         "last_season",
    "draft_year":          "draft_year",
    "draft_club":          "draft_team",
    "draft_round":         "draft_round",
    "draft_number":        "draft_pick",
    # Physical
    "height":              "height",
    "weight":              "weight",
    # Cross-reference IDs (this nflverse build's actual set)
    "pfr_id":              "pfr_id",
    "pff_id":              "pff_id",
    "espn_id":             "espn_id",
    "esb_id":              "esb_id",
    "nfl_id":              "nfl_id",
    "otc_id":              "otc_id",
    "smart_id":            "smart_id",
}

_missing_src = [f"{o}<-{s}" for o, s in _COLMAP.items() if s not in players_raw.columns]
if _missing_src:
    print(f"Note: source columns absent in this nflverse build: {_missing_src}")

dim_nfl_players = pd.DataFrame({
    out_col: (players_raw[src_col] if src_col in players_raw.columns else pd.NA)
    for out_col, src_col in _COLMAP.items()
})

# Drop rows with no gsis_id (registry key)
before = len(dim_nfl_players)
dim_nfl_players = dim_nfl_players[dim_nfl_players["gsis_id"].notna()].reset_index(drop=True)
if before != len(dim_nfl_players):
    print(f"Dropped {before - len(dim_nfl_players)} rows with null gsis_id")

out_path = DATA / "dim_nfl_players.parquet"
dim_nfl_players.to_parquet(out_path, index=False)
print(f"dim_nfl_players: {len(dim_nfl_players):,} players -> {out_path}")
print(f"team_abbr populated: {dim_nfl_players['team_abbr'].notna().mean():.1%} | "
      f"entry_year populated: {dim_nfl_players['entry_year'].notna().mean():.1%}")
dim_nfl_players.head(5)

## 3 -- Validation

In [4]:
df = pd.read_parquet(DATA / "dim_nfl_players.parquet")
print(f"dim_nfl_players: {len(df):,} rows, {len(df.columns)} columns")
print()
print("By status:")
print(df["status"].value_counts().to_string())
print()
print("By position_group (top 10):")
print(df["position_group"].value_counts().head(10).to_string())
print()
# Cross-ref ID coverage
id_cols = ["pfr_id", "espn_id", "sleeper_id", "pff_id", "yahoo_id"]
print("Cross-ref ID coverage:")
for col in id_cols:
    if col in df.columns:
        pct = df[col].notna().mean()
        print(f"  {col:<20} {pct:.1%}")

dim_nfl_players: 24,966 rows, 30 columns

By status:
status
ACT    14713
CUT     3376
RES     3183
DEV     2770
UFA      281
RSN      152
NWT      121
PUP      115
UDF      102
RSR       47
SUS       39
RET       33
RFA       25
EXE        6
INA        3

By position_group (top 10):
position_group
DB      4476
OL      4110
DL      3531
LB      3436
WR      3277
RB      2671
TE      1591
QB      1011
SPEC     863

Cross-ref ID coverage:
  pfr_id               90.3%
  espn_id              66.9%
  sleeper_id           0.0%
  pff_id               45.0%
  yahoo_id             0.0%
